Formatação de Dataframe Unindo os DataSets 

In [ ]:
import pandas as pd
import glob
import os

caminho_pasta = 'dados_inmet'
arquivos = glob.glob(os.path.join(caminho_pasta, "*.CSV"))

lista_final = []

print(f"--- Iniciando Unificação de {len(arquivos)} arquivos ---\n")

for f in arquivos:
    try:
        # 1. Lendo o arquivo bruto
        df_temp = pd.read_csv(f, sep=';', encoding='latin-1', skiprows=8, decimal=',', index_col=False)
        df_temp.columns = df_temp.columns.str.strip().str.upper()
        
        # 2. Localização Dinâmica de Colunas (O segredo para não ter coluna vazia)
        # Procuramos colunas que CONTENHAM os termos, independente do nome completo
        def encontrar_coluna(termos):
            for termo in termos:
                for col in df_temp.columns:
                    if termo in col: return col
            return None

        # Mapeamento
        mapa = {
            'DATA': encontrar_coluna(['DATA']),
            'HORA': encontrar_coluna(['HORA']),
            'CHUVA': encontrar_coluna(['PRECIPITAÇÃO', 'CHUVA']),
            'PRESSAO': encontrar_coluna(['PRESSAO ATMOSFERICA']),
            'TEMP': encontrar_coluna(['TEMPERATURA DO AR - BULBO SECO', 'TEMP_AR']),
            'UMID': encontrar_coluna(['UMIDADE RELATIVA']),
            'VENTO': encontrar_coluna(['RAJADA', 'VENTO_RAJADA']),
            'RAD': encontrar_coluna(['RADIACAO'])
        }

        # 3. Filtragem e Renomeação
        # Removemos os que não foram encontrados (None)
        mapa_limpo = {v: k for k, v in mapa.items() if v is not None}
        df_filtrado = df_temp[list(mapa_limpo.keys())].rename(columns=mapa_limpo)
        
        # 4. Limpeza de Linhas Inúteis
        # Removemos linhas onde a data é nula ou a hora está mal formada
        df_filtrado = df_filtrado.dropna(subset=['DATA', 'HORA'])
        
        #limpeza de linhas vazias 
        limiar_minimo = 4

        #Definimos que a linha precisa de pelo menos 4 valores REAIS para ser mantida
        # (Ex: Data, Hora, Temperatura e Humidade). Se tiver menos que isso, é lixo.
        df_filtrado = df_filtrado.dropna(thresh=limiar_minimo)
        
        #Especificamente para 2025, removemos linhas onde os sensores principais estão vazios
        df_filtrado = df_filtrado.dropna(subset=['CHUVA', 'TEMP'], how='all')

        lista_final.append(df_filtrado)
        print(f"✅ {os.path.basename(f)}: {len(df_filtrado)} linhas e {df_filtrado.columns.tolist()} colunas.")

    except Exception as e:
        print(f"❌ Erro no arquivo {f}: {e}")        

# Juntando tudo no Dataset Mestre
if lista_final:
    df_master = pd.concat(lista_final, axis=0, ignore_index=True)
    
    # Ordenar por data para não ficar bagunçado
    df_master = df_master.sort_values(['DATA', 'HORA'])
    
else:
    print("\nERRO: Nenhum arquivo foi lido corretamente.")

--- Iniciando Unificação de 10 arquivos ---

✅ INMET_NE_CE_A305_FORTALEZA_01-01-2017_A_31-12-2017.CSV: 8760 linhas e ['DATA', 'HORA', 'CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD'] colunas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2018_A_31-12-2018.CSV: 8760 linhas e ['DATA', 'HORA', 'CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD'] colunas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2019_A_31-12-2019.CSV: 6356 linhas e ['DATA', 'HORA', 'CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD'] colunas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2020_A_31-12-2020.CSV: 8777 linhas e ['DATA', 'HORA', 'CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD'] colunas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2021_A_31-12-2021.CSV: 8179 linhas e ['DATA', 'HORA', 'CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD'] colunas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2022_A_31-12-2022.CSV: 8712 linhas e ['DATA', 'HORA', 'CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD'] colunas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2023_A_31-12-2023.CS

Conversão para horário local e formatação de data e hora

In [5]:
#convertendo string para int
df_master['HORA_NUM'] = df_master['HORA'].str.extract(r'(\d+)').astype(int)

#A formatação das horas de 2019 a 2025 são diferentes dos outros anos,
#precisamos fazer a padronização
df_master['HORA_NUM'] = df_master['HORA_NUM'].apply(lambda x: x // 100 if x >= 100 else x)

#Agora precisamos padronizar a coluna 'HORA' para o formato Brasileiro horário local

#retirando do formato UTC e unindo as colunas 'DATA' e 'HORA'
df_master['dt_utc'] = pd.to_datetime(
    df_master['DATA'].str.replace('/', '-') + ' ' +
    df_master['HORA_NUM'].astype(str).str.zfill(2) + ':00'
                                     )
#Convertendo para o fuso horário de fortaleza (UTC-3)
df_master['dt_local'] = df_master['dt_utc'] - pd.Timedelta(hours=3)

#Extraindo as informações locais para treinar o modelo
df_master['HORA_LOCAL'] = df_master['dt_local'].dt.hour
df_master['DATA_LOCAL'] = df_master['dt_local'].dt.date

#Descartando as colunas antigas 'DATA', 'HORA' e as auxiliates 'dt_utc', 'HORA_NUM'
df_master = df_master.drop(columns = ['DATA', 'HORA', 'dt_utc', 'HORA_NUM'])


Formalização de linhas com dados vazios

In [6]:
import numpy as np

# Converter o erro do INMET (-9999) em Valor Nulo Real (NaN)
df_master['RAD'] = df_master['RAD'].replace(-9999, np.nan)

# 2. Preenchimento Inteligente: Noite = Radiação 0
# Considerando que em Fortaleza o sol se põe por volta das 18h e nasce às 05:30h
condicao_noite = (df_master['HORA_LOCAL'] >= 18) | (df_master['HORA_LOCAL'] <= 5)

df_master.loc[condicao_noite, 'RAD'] = df_master.loc[condicao_noite, 'RAD'].fillna(0)

# 3. Agora sim, aplica a interpolação linear apenas para buracos pequenos (máx  imo 2h)
df_master['RAD'] = df_master['RAD'].interpolate(method='linear', limit=2)

# 4. Se ainda sobrarem nulos (buracos grandes), preenchemos com 0 para não quebrar o modelo
# (Ou você pode optar por dropna() se quiser apenas dados perfeitos)
df_master['RAD'] = df_master['RAD'].fillna(0)

Tratamento de Nulos

In [7]:
df_copy = df_master.copy()

# Remover duplicados
df_copy = df_copy.drop_duplicates(subset='dt_local', keep='last')

# Garantir que o DATA_LOCAL É DATATIME
df_copy['DATA_LOCAL'] = pd.to_datetime(df_copy['DATA_LOCAL'])

# Ordenar 
df_copy = df_copy.sort_values('dt_local')

# Colunas
colunas = ['CHUVA','PRESSAO','TEMP','UMID','VENTO','RAD']

# Interpolação
for col in colunas:
    df_copy[col] = df_copy[col].interpolate(limit=6)

# Identificar dias ruins
dias_remover = set()

for col in colunas:
    is_nan = df_copy[col].isna()
    grupos = (is_nan != is_nan.shift()).cumsum()
    
    blocos = df_copy[is_nan].groupby(grupos).size()
    blocos_grandes = blocos[blocos > 6].index
    
    dias = df_copy.loc[grupos.isin(blocos_grandes), 'DATA_LOCAL']
    dias_remover.update(pd.to_datetime(dias).dt.date)

# Remover dias ruins
df_copy = df_copy[~df_copy['DATA_LOCAL'].dt.date.isin(dias_remover)]

# Remover NaN restantes
df_copy = df_copy.dropna()

# Verificar
df_copy.isna().sum()



CHUVA         0
PRESSAO       0
TEMP          0
UMID          0
VENTO         0
RAD           0
dt_local      0
HORA_LOCAL    0
DATA_LOCAL    0
dtype: int64

Criação do Delta

In [8]:
# Ordenar 
df_copy = df_copy.sort_values('dt_local')

# Cria calculo para o DELTA_P e e limita 2 casas decimais
df_copy['DELTA_P'] = (df_copy['PRESSAO'] - df_copy['PRESSAO'].shift(3)).round(2)

# Remover todas as linhas onde a coluna DELTA_P é NaN e reseta o índice
df_copy = df_copy.dropna(subset=['DELTA_P']).reset_index(drop=True)

# Deixar maiúsculo
df_copy = df_copy.rename(columns={'dt_local': 'DT_LOCAL'})

# Mostrar tabela final
df_copy

,CHUVA,PRESSAO,TEMP,UMID,VENTO,RAD,DT_LOCAL,HORA_LOCAL,DATA_LOCAL,DELTA_P
0,0.0,1008.7,27.4,76.0,7.0,0.0,2017-01-01 00:00:00,0,2017-01-01,-0.3
1,0.0,1007.8,27.1,76.0,7.0,0.0,2017-01-01 01:00:00,1,2017-01-01,-1.3
2,0.0,1007.5,27.0,75.0,6.6,0.0,2017-01-01 02:00:00,2,2017-01-01,-1.6
3,0.0,1007.2,26.8,76.0,6.6,0.0,2017-01-01 03:00:00,3,2017-01-01,-1.5
4,0.0,1007.1,26.7,77.0,5.3,0.0,2017-01-01 04:00:00,4,2017-01-01,-0.7
...,...,...,...,...,...,...,...,...,...,...
59905,0.0,1009.7,31.2,60.0,7.0,1.3,2025-01-30 11:00:00,11,2025-01-30,2.6
59906,0.0,1009.3,31.8,55.0,7.3,0.0,2025-01-30 12:00:00,12,2025-01-30,-0.4
59907,0.0,1008.1,31.8,56.0,7.5,0.0,2025-01-30 13:00:00,13,2025-01-30,-1.7
59908,0.0,1006.7,31.3,57.0,7.1,0.0,2025-01-30 14:00:00,14,2025-01-30,-3.0


Exportando Dados

In [ ]:
#Instala o dataset com os dados formalizados em formato csv
df_copy.to_csv('dados_formatados/FORTALEZA_DADOS_CLIMATICOS.csv', sep=';', index=False, encoding='utf-8')
print(f"\nSUCESSO! Total de {len(df_copy)} linhas consolidadas em 'FORTALEZA_DATASET_CONSOLIDADO'")


SUCESSO! Total de 59910 linhas consolidadas em 'FORTALEZA_DATASET_CONSOLIDADO'


In [ ]:
import pandas as pd

# Base escolhida: InfoDengue
# Cidade analisada: Fortaleza
# Período da consulta: 2017 até 2024

geocode = 2304400
doenca = "dengue"
ano_inicial = 2017
ano_final = 2024
semana_inicial = 1
semana_final = 53

arquivo_csv = "infodengue_fortaleza_2017_2024.csv"
arquivo_relatorio = "relatorio_prospeccao_validacao.txt"

# montando a url da API
url = (
    "https://info.dengue.mat.br/api/alertcity?"
    f"geocode={geocode}"
    f"&disease={doenca}"
    f"&format=csv"
    f"&ew_start={semana_inicial}"
    f"&ew_end={semana_final}"
    f"&ey_start={ano_inicial}"
    f"&ey_end={ano_final}"
)

print("URL usada:")
print(url)

# leitura da base
df = pd.read_csv(url)

print("\nBase carregada com sucesso")
print("Dimensão da base:", df.shape)

print("\nPrimeiras linhas:")
print(df.head())

print("\nColunas encontradas:")
print(df.columns.tolist())

# convertendo a coluna de data
df["data_iniSE"] = pd.to_datetime(df["data_iniSE"], errors="coerce")

# salvando o csv baixado
df.to_csv(arquivo_csv, index=False, encoding="utf-8")
print(f"\nCSV salvo com sucesso: {arquivo_csv}")

print("\n--- VALIDAÇÃO DAS COLUNAS ESSENCIAIS ---")

colunas_essenciais = ["data_iniSE", "SE", "municipio_nome", "casos"]
faltando = []

for coluna in colunas_essenciais:
    if coluna not in df.columns:
        faltando.append(coluna)

if len(faltando) == 0:
    print("Todas as colunas essenciais estão presentes.")
else:
    print("Colunas faltando:", faltando)

print("\n--- VALIDAÇÃO DA LOCALIDADE ---")

municipios = df["municipio_nome"].dropna().unique()
print("Municípios encontrados:")
print(municipios)

if len(municipios) == 1 and municipios[0] == "Fortaleza":
    print("A base está filtrada apenas para Fortaleza.")
else:
    print("A base precisa de revisão no filtro de município.")

print("\n--- VALIDAÇÃO TEMPORAL ---")

data_inicial = df["data_iniSE"].min()
data_final = df["data_iniSE"].max()

print("Data inicial:", data_inicial)
print("Data final:", data_final)

print("\n--- VALIDAÇÃO DA GRANULARIDADE ---")

if "data_iniSE" in df.columns and "SE" in df.columns:
    print("A base possui data e semana epidemiológica.")
    print("Isso permite o cruzamento com os dados climáticos.")
else:
    print("A base não possui a granularidade necessária.")

print("\n--- CHECAGEM DE QUALIDADE ---")

nulos = df.isnull().sum().sort_values(ascending=False)
print("\nColunas com mais valores nulos:")
print(nulos.head(15))

duplicadas = df.duplicated().sum()
print("\nQuantidade de linhas duplicadas:", duplicadas)

print("\nNulos nas colunas essenciais:")
for coluna in colunas_essenciais:
    if coluna in df.columns:
        print(f"{coluna}: {df[coluna].isnull().sum()}")

# gerando o relatório
relatorio = []

relatorio.append("PROSPECÇÃO E VALIDAÇÃO DE DATASET EPIDEMIOLÓGICO\n")
relatorio.append("=" * 60 + "\n\n")

relatorio.append("Fonte analisada: InfoDengue\n")
relatorio.append("Município: Fortaleza\n")
relatorio.append(f"Doença: {doenca}\n")
relatorio.append(f"Período consultado: {ano_inicial} a {ano_final}\n\n")

relatorio.append("1. Compatibilidade de datas\n")
relatorio.append(f"- Data inicial encontrada: {data_inicial}\n")
relatorio.append(f"- Data final encontrada: {data_final}\n")
relatorio.append("- A base cobre o intervalo necessário para comparação com o clima.\n\n")

relatorio.append("2. Granularidade\n")
if "data_iniSE" in df.columns and "SE" in df.columns:
    relatorio.append("- A base possui data de início da semana e semana epidemiológica.\n")
    relatorio.append("- Isso permite fazer correlação com dados climáticos.\n\n")
else:
    relatorio.append("- A base não possui a granularidade necessária.\n\n")

relatorio.append("3. Integridade\n")
if len(faltando) == 0:
    relatorio.append("- Todas as colunas essenciais estão presentes.\n")
else:
    relatorio.append(f"- Colunas faltando: {faltando}\n")

relatorio.append(f"- Quantidade de linhas duplicadas: {duplicadas}\n")
relatorio.append("- Nulos nas colunas essenciais:\n")
for coluna in colunas_essenciais:
    if coluna in df.columns:
        relatorio.append(f"  - {coluna}: {df[coluna].isnull().sum()}\n")
relatorio.append("\n")

relatorio.append("4. Localidade\n")
if len(municipios) == 1 and municipios[0] == "Fortaleza":
    relatorio.append("- A base contém apenas registros de Fortaleza.\n\n")
else:
    relatorio.append("- A base precisa de revisão no filtro de localidade.\n\n")

relatorio.append("5. Base final escolhida\n")
relatorio.append(
    "- A base final escolhida foi a do InfoDengue para Fortaleza, "
    "porque possui recorte municipal, série histórica adequada, "
    "semana epidemiológica e leitura direta no Pandas.\n"
)

with open(arquivo_relatorio, "w", encoding="utf-8") as arquivo:
    arquivo.writelines(relatorio)

print(f"\nRelatório salvo com sucesso: {arquivo_relatorio}")
print("\nProcesso concluído.")

URL usada:
https://info.dengue.mat.br/api/alertcity?geocode=2304400&disease=dengue&format=csv&ew_start=1&ew_end=53&ey_start=2017&ey_end=2024

Base carregada com sucesso.
Quantidade de linhas e colunas: (417, 31)

Primeiras linhas:
   data_iniSE      SE  casos_est  casos_est_min  casos_est_max  casos  \
0  2024-12-22  202452       42.0             42             42     42   
1  2024-12-15  202451       65.0             65             65     65   
2  2024-12-08  202450       82.0             82             82     82   
3  2024-12-01  202449       85.0             85             85     85   
4  2024-11-24  202448       99.0             99             99     99   

      p_rt1  p_inc100k  Localidade_id  nivel  ...    umidmed    umidmin  \
0  0.000549   1.617776              0      1  ...  75.053529  57.029057   
1  0.027220   2.503701              0      1  ...  78.271629  61.219586   
2  0.075172   3.158515              0      1  ...  75.892571  59.095457   
3  0.039801   3.274070        